# Flowchart Visual Test

Visualizzazione interattiva del parser flowchart su applet reali.
- Carica il dataset `applets_synt_new_final.jsonl`
- Esegue L1 validation → display labels → flowchart HTML
- Mostra side-by-side **non-expert** vs **expert**
- Filtri per tipo (guard+action, skip only, setter only, multi-path)

In [1]:
import json, sys
from pathlib import Path
from IPython.display import display, HTML
import ipywidgets as widgets
from tqdm import tqdm

# project root
ROOT = Path.cwd().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.code_parsing.feedback import run_l1_validation
from src.code_parsing.catalog_validator import build_display_labels
from src.code_parsing.flowchart import render_flowchart_html

DATASET       = ROOT / "data" / "dataset" / "applets" / "applets_synt_new_final.jsonl"
TRIGGERS_PATH = ROOT / "data" / "ifttt_catalog" / "triggers.json"
ACTIONS_PATH  = ROOT / "data" / "ifttt_catalog" / "actions.json"

print(f"Root: {ROOT}")
print(f"Dataset: {DATASET.exists()}")

Root: C:\Users\ldesa\PycharmProjects\TAP basic
Dataset: True


## 1. Caricamento dataset e catalogo

In [2]:
with open(DATASET, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

with open(TRIGGERS_PATH, "r", encoding="utf-8") as f:
    ALL_TRIGGERS = json.load(f)
with open(ACTIONS_PATH, "r", encoding="utf-8") as f:
    ALL_ACTIONS = json.load(f)

TRIGGER_INDEX = {t["api_endpoint_slug"]: t for t in ALL_TRIGGERS}
ACTION_INDEX  = {a["api_endpoint_slug"]: a for a in ALL_ACTIONS}

print(f"Records: {len(records)}")
print(f"Triggers catalog: {len(ALL_TRIGGERS)}")
print(f"Actions catalog: {len(ALL_ACTIONS)}")

Records: 2246
Triggers catalog: 3620
Actions catalog: 2491


## 2. Pipeline: L1 → display labels → flowchart

In [3]:
LANG = "it"

def process_record(rec, lang=LANG):
    """Run full pipeline on one record. Returns dict or None."""
    code = rec.get("filter_code", "")
    trigger_slugs = rec.get("trigger_apis", [])
    action_slugs  = rec.get("action_apis", [])

    l1 = run_l1_validation(code=code, trigger_slugs=trigger_slugs,
                           action_slugs=action_slugs, lang=lang)
    if not l1.syntax_ok or not l1.outcomes_raw:
        return None

    triggers = [TRIGGER_INDEX[s] for s in trigger_slugs if s in TRIGGER_INDEX]
    actions  = [ACTION_INDEX[s]  for s in action_slugs  if s in ACTION_INDEX]
    dl = build_display_labels(triggers, actions, trigger_slugs, action_slugs)

    has_skip    = any(o.get("skip") for o in l1.outcomes_raw)
    has_setters = any(not o.get("skip") for o in l1.outcomes_raw)

    return {
        "idx": rec.get("row_index", "?"),
        "name": rec.get("name", ""),
        "desc": rec.get("rule_description", rec.get("description", "")),
        "code": code,
        "n_outcomes": len(l1.outcomes_raw),
        "has_skip": has_skip,
        "has_setters": has_setters,
        "is_mixed": has_skip and has_setters,
        "outcomes_raw": l1.outcomes_raw,
        "display_labels": dl,
        "l1": l1,
    }

In [4]:
%%time
results = []
from tqdm import tqdm
for  rec in tqdm(records):
    r = process_record(rec)
    if r is not None:
        results.append(r)

n_mixed  = sum(1 for r in results if r["is_mixed"])
n_skip   = sum(1 for r in results if r["has_skip"] and not r["has_setters"])
n_setter = sum(1 for r in results if r["has_setters"] and not r["has_skip"])
n_multi  = sum(1 for r in results if r["n_outcomes"] >= 3)

print(f"Parsed: {len(results)} / {len(records)}")
print(f"  Mixed (guard+action): {n_mixed}")
print(f"  Skip only:            {n_skip}")
print(f"  Setter only:          {n_setter}")
print(f"  3+ paths:             {n_multi}")

100%|██████████| 2246/2246 [04:43<00:00,  7.94it/s]

Parsed: 2243 / 2246
  Mixed (guard+action): 955
  Skip only:            510
  Setter only:          778
  3+ paths:             64
CPU times: total: 4min 27s
Wall time: 4min 43s


## 3. Viewer interattivo

Usa i filtri per selezionare il tipo di applet, poi naviga con lo slider.

In [5]:
def render_side_by_side(r, lang=LANG):
    """Render non-expert and expert flowcharts side by side."""
    html_ne = render_flowchart_html(
        r["outcomes_raw"], lang=lang, user_type="non_expert",
        display_labels=r["display_labels"],
    )
    html_ex = render_flowchart_html(
        r["outcomes_raw"], lang=lang, user_type="expert",
        display_labels=r["display_labels"],
    )
    import html as htmlmod
    srcdoc_ne = htmlmod.escape(html_ne)
    srcdoc_ex = htmlmod.escape(html_ex)

    # type tag
    if r["is_mixed"]:
        tag = '<span style="background:#fff3e0;color:#e65100;padding:2px 8px;border-radius:10px;font-size:11px;font-weight:700">GUARD + ACTION</span>'
    elif r["has_skip"]:
        tag = '<span style="background:#fce4ec;color:#c62828;padding:2px 8px;border-radius:10px;font-size:11px;font-weight:700">SKIP ONLY</span>'
    else:
        tag = '<span style="background:#e8f5e9;color:#2e7d32;padding:2px 8px;border-radius:10px;font-size:11px;font-weight:700">SETTER ONLY</span>'

    paths_tag = f'<span style="background:#e3f2fd;color:#1565c0;padding:2px 8px;border-radius:10px;font-size:11px;font-weight:700">{r["n_outcomes"]} PATHS</span>'

    return f"""
    <div style="font-family:system-ui,sans-serif;margin-bottom:8px">
      <div style="display:flex;gap:8px;align-items:baseline;margin-bottom:4px">
        <span style="font-size:11px;color:#888;font-family:monospace">#{r['idx']}</span>
        <strong style="font-size:14px">{htmlmod.escape(r['name'][:80])}</strong>
        {tag} {paths_tag}
      </div>
      <div style="font-size:12px;color:#555;margin-bottom:10px">{htmlmod.escape(r['desc'][:200])}</div>
      <div style="display:flex;gap:12px">
        <div style="flex:1">
          <div style="font-size:10px;font-weight:700;color:#888;text-transform:uppercase;margin-bottom:4px">Non-Expert</div>
          <iframe srcdoc="{srcdoc_ne}" style="width:100%;border:1px solid #e0e0e0;border-radius:6px;min-height:300px"
                  onload="this.style.height=this.contentDocument.documentElement.scrollHeight+'px'"></iframe>
        </div>
        <div style="flex:1">
          <div style="font-size:10px;font-weight:700;color:#888;text-transform:uppercase;margin-bottom:4px">Expert</div>
          <iframe srcdoc="{srcdoc_ex}" style="width:100%;border:1px solid #e0e0e0;border-radius:6px;min-height:300px"
                  onload="this.style.height=this.contentDocument.documentElement.scrollHeight+'px'"></iframe>
        </div>
      </div>
    </div>
    """


# ── Widgets ──
filter_dropdown = widgets.Dropdown(
    options=["All", "Mixed (guard+action)", "Skip only", "Setter only", "3+ paths"],
    value="All",
    description="Filtro:",
)
index_slider = widgets.IntSlider(value=0, min=0, max=0, description="#", continuous_update=False)
output = widgets.Output()

filtered = list(results)  # current filtered list

def apply_filter(change=None):
    global filtered
    f = filter_dropdown.value
    if f == "All":
        filtered = results
    elif f == "Mixed (guard+action)":
        filtered = [r for r in results if r["is_mixed"]]
    elif f == "Skip only":
        filtered = [r for r in results if r["has_skip"] and not r["has_setters"]]
    elif f == "Setter only":
        filtered = [r for r in results if r["has_setters"] and not r["has_skip"]]
    elif f == "3+ paths":
        filtered = [r for r in results if r["n_outcomes"] >= 3]
    index_slider.max = max(len(filtered) - 1, 0)
    index_slider.value = 0
    show_current()

def show_current(change=None):
    output.clear_output(wait=True)
    with output:
        if not filtered:
            display(HTML("<p>Nessun risultato per questo filtro.</p>"))
            return
        i = min(index_slider.value, len(filtered) - 1)
        display(HTML(f"<div style='font-size:12px;color:#888;margin-bottom:6px'>{i+1} / {len(filtered)}</div>"))
        display(HTML(render_side_by_side(filtered[i])))

filter_dropdown.observe(apply_filter, names="value")
index_slider.observe(show_current, names="value")

apply_filter()  # init
display(widgets.HBox([filter_dropdown, index_slider]))
display(output)

Output()

## 4. Statistiche strutturali

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"idx": r["idx"], "name": r["name"], "n_outcomes": r["n_outcomes"],
     "has_skip": r["has_skip"], "has_setters": r["has_setters"],
     "is_mixed": r["is_mixed"],
     "n_or_clauses": max(
         len(__import__('src.code_parsing.flowchart', fromlist=['_split_top_level_or'])
             ._split_top_level_or(o.get('condition', '')))
         for o in r["outcomes_raw"]
     ),
    }
    for r in results
])

print("=== Distribuzione outcomes per applet ===")
print(df["n_outcomes"].value_counts().sort_index())
print()
print("=== Tipo di applet ===")
print(f"  Mixed (guard+action): {df['is_mixed'].sum()}")
print(f"  Skip only:            {(df['has_skip'] & ~df['has_setters']).sum()}")
print(f"  Setter only:          {(df['has_setters'] & ~df['has_skip']).sum()}")
print()
print("=== Max OR clauses nella condizione più complessa ===")
print(df["n_or_clauses"].describe())
print()
print("Applet con 3+ clausole OR:")
complex_or = df[df["n_or_clauses"] >= 3].sort_values("n_or_clauses", ascending=False)
print(complex_or[["idx", "name", "n_outcomes", "n_or_clauses", "is_mixed"]].head(20).to_string(index=False))

## 5. Galleria: casi più complessi

I 10 applet con più clausole OR (condizioni split).

In [ ]:
# Sort by max OR clauses, show top 10
by_complexity = sorted(results, key=lambda r: max(
    len(__import__('src.code_parsing.flowchart', fromlist=['_split_top_level_or'])
        ._split_top_level_or(o.get('condition', '')))
    for o in r["outcomes_raw"]
), reverse=True)

for r in by_complexity[:10]:
    display(HTML(render_side_by_side(r)))
    display(HTML('<hr style="margin:20px 0">'))